# 00 — The Big Picture: Control and Machine Learning

**Mathematical Foundations of Control and Machine Learning — Winter 2026**

> **Central question.** *What do controlling a dynamical system and training a learning machine have in common?*

This notebook is the map of the course: what the two fields are, where they meet, and how the notebooks are organized. A glossary of recurring terms is collected in the appendix.

### Table of Contents

1. **What the course connects** — control theory and machine learning
2. **The map of the course** — the five parts
3. **How the notebooks are built** — the shared skeleton
4. **A first example** — the harmonic oscillator, free and with feedback
5. **Exercises**

**Appendix A.** Glossary

In [1]:
%matplotlib inline
import torch
import plotting as P

torch.manual_seed(0);

In [2]:
# RK4 integrator for x' = f(t, x) along the time grid ts
def odeint(f, x0, ts):
    xs = [x0]
    for t, t1 in zip(ts[:-1], ts[1:]):
        x, h = xs[-1], t1 - t
        k1 = f(t, x)
        k2 = f(t + h / 2, x + h / 2 * k1)
        k3 = f(t + h / 2, x + h / 2 * k2)
        k4 = f(t + h, x + h * k3)
        xs.append(x + h / 6 * (k1 + 2 * k2 + 2 * k3 + k4))
    return torch.stack(xs)

## 1. What the course connects

**Control.** A system has a *state* that evolves according to a law, the *state equation*, in which we can act through a *control*. The goal is to drive the state close to a desired one. Three questions organize the field:

- **optimal control:** which admissible control minimizes a cost, such as the distance to a target?
- **controllability:** can the state be driven *exactly* to a prescribed target?
- **stabilization / feedback:** can we find a rule $u=F(x)$, computed in real time from the state, that drives the system towards equilibrium?

**Machine learning.** A model with parameters $w$ is fitted to data by minimizing a loss $F(w)$, with the aim of making good predictions on *unobserved* data.

**Where they meet.** 

$$
\begin{aligned}
x'(t)&=w(t)\,\sigma\big(a(t)\cdot x(t)+b(t)\big),\\
x_{k+1}&=x_k+h\,w_k\,\sigma(a_k\cdot x_k+b_k),
\end{aligned}
$$

The first is a **controlled dynamical system** (a *neural ODE*): a neuron $w\,\sigma(a\cdot x+b)$ is used as the vector field, and the parameters $(w,a,b)$ act as controls. The second is one step of the explicit Euler method for it, and it is exactly the layer of a **residual neural network (ResNet)**. For the two dynamical models, training means choosing the controls that minimize a cost along the trajectories of the data points.

The course therefore runs along one line: first control (Part I), then optimization and long-time control (Part II), then machine learning (Part III), then networks read as dynamical systems (Part IV), and finally reinforcement learning and a synthesis of the two fields (Part V).

## 2. The map of the course

| Part | Notebooks |
|---|---|
| I. Control | [01 What is control?](01_control_fundamentals.ipynb) · [02 Linear control and the Kalman condition](02_linear_control_kalman.ipynb) · [03 Observability, duality and the construction of controls](03_observability_duality_adjoints.ipynb) · [04 Bang-bang and switching controls](04_bangbang_switching.ipynb) · [05 Feedback stabilization, LQR and MPC](05_feedback_stabilization.ipynb) |
| II. Optimization and long-time control | [06 Minimizers, gradient flows and gradient descent](06_variational_principles_gradient_flows.ipynb) · [07 Adjoint methods and the heat equation](07_adjoints_density_heat_control.ipynb) · [08 The turnpike property](08_long_time_control_turnpike.ipynb) |
| III. Machine learning | [09 Learning from data and generalization](09_machine_learning_foundations_generalization.ipynb) · [10 Training: gradient descent and SGD](10_training_gd_sgd.ipynb) · [11 Optimization algorithms: momentum and adaptive methods](11_optimization.ipynb) |
| IV. Networks as dynamical systems | [12 Shallow networks: representation and sparsity](12_shallow_networks.ipynb) · [13 ResNets and neural ODEs](13_neuralodes.ipynb) · [14 Classification as simultaneous control](14_classification_simultaneous_control.ipynb) · [15 Transformers](15_transformers.ipynb) · [16 Neural transport](16_neural_transport.ipynb) · [17 Diffusion models](17_diffusion_models.ipynb) |
| V. Closing the loop | [18 Reinforcement learning](18_reinforcement_learning.ipynb) · [19 Control and learning: a synthesis](19_synthesis_control_and_learning.ipynb) |


## 3. How the notebooks are built

- a title, an overview of the sections, and a short setup cell;
- the mathematics carries the argument: definitions, theorems and examples with proofs;
- the code cells are used to illustrate and verify the mathematical concepts;
- **exercises**, graded ★ (conceptual), ★★ (mathematical) and ★★★ (computational);
- references.

Side material that would interrupt the main line goes into an appendix placed before the references.

## 4. A first example: the harmonic oscillator

A mass on a spring, pushed by a force $u$, obeys $x_1''+x_1=u$. With the state $x=(x_1,x_2)=(\text{position},\text{velocity})$ this is the linear control system

$$
x'=Ax+Bu,\qquad A=\begin{pmatrix}0&1\\-1&0\end{pmatrix},\quad B=\begin{pmatrix}0\\1\end{pmatrix},\qquad x^0=(1,0).
$$

In [3]:
A = torch.tensor([[0.0, 1.0], [-1.0, 0.0]])
B = torch.tensor([[0.0], [1.0]])
x0 = torch.tensor([1.0, 0.0])
ts = torch.linspace(0, 12, 481)

We compare the free motion, $u=0$, with the feedback $u=-x_2$, which pushes against the velocity.

In [4]:
free = odeint(lambda t, x: A @ x, x0, ts)                         # u = 0
damped = odeint(lambda t, x: A @ x + B @ (-x[1:]), x0, ts)       # u = -x2

In [6]:
idx = torch.arange(0, len(ts), 6)                                # 81 frames
fig, (axm, axp) = P.panels(2, ratios=[1.3, 1], figsize=(7.6, 3.2))
move_free = P.spring_mass_artist(axm, 0.8, P.GREY, "$u=0$")
move_damped = P.spring_mass_artist(axm, -0.8, P.BLUE, "$u=-x_2$")
P.label(axm, xlim=(-3.0, 2.3), ylim=(-1.8, 1.8), equal=True, off=True)

axp.plot(0, 0, "*", color=P.ORANGE, ms=10)
paths = [axp.plot([], [], color=c, lw=1.4)[0] for c in (P.GREY, P.BLUE)]
dots = [axp.plot([], [], "o", ms=5, color=c)[0] for c in (P.GREY, P.BLUE)]
P.label(axp, "position $x_1$", "velocity $x_2$", xlim=(-1.2, 1.2), ylim=(-1.2, 1.2), equal=True)

def update(i):
    k = idx[i]
    move_free(free[k, 0]); move_damped(damped[k, 0])
    for path, dot, x in zip(paths, dots, (free, damped)):
        path.set_data(x[:k + 1, 0], x[:k + 1, 1])
        dot.set_data(x[k:k + 1, 0], x[k:k + 1, 1])
    axp.set_title(f"$t={ts[k]:.1f}$")

P.animate(fig, update, len(idx), interval=60)

## Appendix A. Glossary

Each term is defined precisely where it is first used; here they are collected for quick reference.

**Dynamical systems and control**

| Term | Meaning |
|---|---|
| state $x(t)\in\mathbb R^n$ | the variables that describe the system at time $t$ |
| control $u(t)\in\mathbb R^m$ | the input we choose to act on the system, taken from a set of *admissible controls* |
| state equation | the law of evolution $x'=f(x,u)$, $x(0)=x^0$; *linear* when $x'=Ax+Bu$ |
| trajectory, horizon $T$ | the solution $t\mapsto x(t)$ on the time interval $[0,T]$ |
| controllability | for every initial state $x^0$ and target $x^1$ there is a control with $x(T)=x^1$ |
| reachable set $\mathcal R(T,x^0)$ | all states $x(T)$ obtained from $x^0$ with admissible controls |
| Kalman matrix | $\mathcal K(A,B)=[B,\ AB,\ \dots,\ A^{n-1}B]$; $x'=Ax+Bu$ is controllable iff $\operatorname{rank}\mathcal K=n$ |
| observability | the initial state can be recovered from an observation $y(t)=Cx(t)$ on $[0,T]$ |
| adjoint system | a system solved backwards in time, $-p'=A^*p$ in the linear case; it gives the gradient of a cost with respect to the control |
| cost functional $J(u)$ | the quantity to be minimized, typically a distance to a target plus the effort $\int_0^T\lvert u\rvert^2dt$ |
| optimal control | a minimizer of $J$ over the admissible controls |
| bang-bang control | a control that only takes extreme values of the admissible set, switching between them |
| feedback, closed loop | a control computed from the current state, $u=F(x)$, and the system that results |
| stabilization | the choice of a feedback that makes the state return to rest |
| LQR | the linear feedback that minimizes a quadratic cost on an infinite horizon |
| MPC | at every step, solve a finite-horizon optimal control problem and apply only its first part |
| turnpike | on long horizons, optimal trajectories stay close to an optimal steady state, except near $t=0$ and $t=T$ |

**Optimization**

| Term | Meaning |
|---|---|
| minimizer | a point $w^*$ with $F(w^*)\le F(w)$ for all $w$ |
| convexity | $F$ lies below its chords; every local minimizer is then global |
| gradient flow | $w'(t)=-\nabla F(w(t))$, continuous-time steepest descent |
| gradient descent | $w_{k+1}=w_k-\eta\nabla F(w_k)$, the explicit Euler scheme of the gradient flow with *learning rate* $\eta$ |
| stochastic gradient descent (SGD) | gradient descent with $\nabla F$ replaced by the gradient on a random *mini-batch* |

**Learning from data**

| Term | Meaning |
|---|---|
| dataset | samples $(x_i,y_i)$, $i=1,\dots,N$: inputs (*features*) and outputs (*labels*) |
| supervised, unsupervised, reinforcement learning | learning from input–output pairs, from inputs alone, and from the outcomes of one's own actions |
| model | a parameterized family of functions $x\mapsto\hat y(x;w)$, $w\in\mathbb R^D$ |
| loss $\ell$ | the penalty $\ell(\hat y,y)$ for predicting $\hat y$ when the true output is $y$ |
| empirical risk | $F(w)=\frac1N\sum_{i=1}^N\ell\big(\hat y(x_i;w),y_i\big)$, the average loss on the training data |
| population risk | $\mathcal R(w)=\mathbb E_{(x,y)\sim\rho}\,\ell\big(\hat y(x;w),y\big)$, the average loss under the unknown data distribution $\rho$ |
| training, validation and test sets | the data used to fit the parameters, to choose between models, and to report the final performance |
| generalization, overfitting | $\mathcal R(w)-F(w)$ is small; overfitting is a small $F(w)$ with a large $\mathcal R(w)$ |
| regularization | a term added to the loss, such as $\alpha\lvert w\rvert^2$, that penalizes complex models |
| policy | in reinforcement learning, the rule that chooses an action in each state |

**Neural networks**

| Term | Meaning |
|---|---|
| neuron, activation | the map $x\mapsto w\,\sigma(a\cdot x+b)$, with a nonlinear *activation* $\sigma$ such as the ReLU $\sigma(z)=\max\{z,0\}$ |
| shallow network | a sum of neurons, $\sum_{j=1}^Kw_j\,\sigma(a_j\cdot x+b_j)$ |
| residual network (ResNet) | a composition of layers $x_{k+1}=x_k+h\,w_k\,\sigma(a_k\cdot x_k+b_k)$ |
| neural ODE | the continuous-time model $x'(t)=w(t)\,\sigma\big(a(t)\cdot x(t)+b(t)\big)$ |
| backpropagation | computing the gradient of the loss from the last layer back to the first |
| attention, transformer | attention updates each token by a data-dependent weighted average of all tokens; a transformer alternates attention and neuron layers |

## References

1. W. E, [*A proposal on machine learning via dynamical systems*](https://doi.org/10.1007/s40304-017-0103-z), Communications in Mathematics and Statistics **5** (2017), 1–11.
2. E. Haber, L. Ruthotto, [*Stable architectures for deep neural networks*](https://doi.org/10.1088/1361-6420/aa9a90), Inverse Problems **34** (2018), 014004.
3. D. Ruiz-Balet, E. Zuazua, [*Neural ODE control for classification, approximation, and transport*](https://doi.org/10.1137/21M1411433), SIAM Review **65** (2023), 735–773.
4. B. Geshkovski, E. Zuazua, [*Turnpike in optimal control of PDEs, ResNets, and beyond*](https://doi.org/10.1017/S0962492922000046), Acta Numerica **31** (2022), 135–263.